# 18j Decoding a new person: one combined system (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g), and the outputs of 09m.

Development analysis, not a prespecified test. One person held out at a time, 200-way identification, centred. The personal map on DINOv2 borrowed predictions
combined with the person's own ridge model on AlexNet features (mixed), against the AlexNet combination and an
all-DINOv2 combination, at k = 250, 500 and 1,000. Secondary by construction: it combines choices made after 18h's
results were seen.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '18j-decoding', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [250, 500, 1000], "draws": [13, 14, 15], "features": ["alexnet", "dinov2"], "stacks": {"alexnet": [["map", "alexnet"], ["own", "alexnet"]], "dinov2": [["map", "dinov2"], ["own", "dinov2"]], "mixed": [["map", "dinov2"], ["own", "alexnet"]]}}')
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']
TAG = '18j'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import plugin_map as pm
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
n_ch, n_t = 17, len(times)
N_TRAIN = len(next(iter(y_train.values())))
win = ha.window_index(times, 100)
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
post = np.repeat((times >= -1e-9)[None, :], n_ch, 0).ravel()

def sources(ztr, zte):
    tr = np.empty((len(people), len(ztr), n_ch * n_t), np.float32)
    te_ = np.empty((len(people), len(zte), n_ch * n_t), np.float32)
    q = np.concatenate([ztr, zte])
    for i, s in enumerate(people):
        pred = pl.ols_paper(ztr, y_train[s].reshape(len(ztr), -1), q)
        tr[i], te_[i] = pred[:len(ztr)], pred[len(ztr):]
    return tr, te_

def similarity(pred, meas):
    # centred over the 200 test images, then Pearson over the post-onset features
    p, m = pred[:, post] - pred[:, post].mean(0), meas[:, post] - meas[:, post].mean(0)
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]  # (measured, candidates)

def calibrate(r, how):
    if how == "none":
        return r
    if how == "csls":  # cross-domain similarity local scaling (Conneau et al. 2018), k = 10, as SATTC's geometric expert
        rm = np.sort(r, 1)[:, -10:].mean(1)
        rp = np.sort(r, 0)[-10:, :].mean(0)
        return 2 * r - rm[:, None] - rp[None, :]
    if how == "dual_z":  # each candidate's similarities standardized over the test responses, then each response's
        z = (r - r.mean(0)) / (r.std(0) + 1e-30)
        return (z - z.mean(1, keepdims=True)) / (z.std(1, keepdims=True) + 1e-30)
    raise ValueError(how)

def test_time_map(pred, meas):
    # label-free, transductive (the 200 test responses as a batch, no pairing used): per 100 ms window, re-mix the
    # prediction's channels so that their covariance over images and time points matches the measured responses'
    from scipy.linalg import sqrtm
    p3, m3 = t3(pred, len(pred)), t3(meas, len(meas))
    out = p3.copy()
    for w in np.unique(win):
        cols = win == w
        xp = (p3[:, :, cols] - p3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        xm = (m3[:, :, cols] - m3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        cp = np.cov(xp, rowvar=False) + 1e-6 * np.eye(n_ch)
        cm = np.cov(xm, rowvar=False) + 1e-6 * np.eye(n_ch)
        a = np.real(np.linalg.inv(sqrtm(cp)) @ sqrtm(cm))
        mean = p3[:, :, cols].mean(0)
        out[:, :, cols] = np.einsum("nct,cd->ndt", p3[:, :, cols] - mean, a) + mean
    return out.reshape(len(pred), -1)

def encoding_r(pred, meas):
    # per-output correlation across the 200 test images with the repetition average, 100-200 ms, averaged
    sel = np.repeat(((times >= 0.1) & (times < 0.2))[None, :], n_ch, 0).ravel()
    p, m = pred[:, sel] - pred[:, sel].mean(0), meas[:, sel] - meas[:, sel].mean(0)
    return float(np.mean((p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)))

def accuracy(r):
    rank = (r > np.diag(r)[:, None]).sum(1)
    return float((rank < 1).mean() * 100), float((rank < 5).mean() * 100)

def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x))}

ALPHAS = [10.0 ** e for e in range(-2, 9)]
F = 5
N_TEST = len(z_test)

def parts_for(person, idx, d, ztr, zte, avg_train, avg_test):
    # the map and own-model parts: returns {"map": (out-of-fold, test), "own": (out-of-fold, test)}, each (n, channels, times)
    k = len(idx)
    target = np.asarray(y_train[person][idx], np.float64)
    flat = target.reshape(k, -1)
    a_k = t3(avg_train[idx], k)
    map_test = h4b.apply_map(h4b.fit_map(a_k, target, win, seed=d)[0], t3(avg_test, N_TEST), win)
    own = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(ztr[idx], flat)
    own_test = t3(own.predict(zte), N_TEST)
    own_oof = t3(pm.out_of_fold(lambda tr, he: own.refit(ztr[idx[tr]], flat[tr]).predict(ztr[idx[he]]), k, F, d), k)
    map_oof = pm.out_of_fold(lambda tr, he: h4b.apply_map(h4b.fit_map(a_k[tr], target[tr], win, seed=d)[0], a_k[he], win), k, F, d)
    return target, {"map": (map_oof, map_test), "own": (own_oof, own_test)}

def stack(target, chosen):
    # the stack: one weight per part for each channel and 100 ms window, fitted on out-of-fold predictions
    w = pm.stack_fit([oof for oof, _ in chosen], target, win)
    return pm.stack_apply(w, [test for _, test in chosen], win).reshape(N_TEST, -1)

def stack_with_mean(target, chosen, intercept):
    # 18n: stack_fit works on image-centred data, so apply it the same way: centre each part's test
    # predictions on that part's mean over the k labelled images, combine, then add an intercept (channels, times)
    w = pm.stack_fit([oof for oof, _ in chosen], target, win)
    out = pm.stack_apply(w, [test - oof.mean(0) for oof, test in chosen], win) + intercept
    return out.reshape(N_TEST, -1)

def similarity_raw(pred, meas):  # the authors' uncentred identification
    p, m = pred[:, post], meas[:, post]
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]
def feature_r(pred, resp):
    # per channel-time correlation across images (centred over images); pred, resp (n, features)
    p, m = pred - pred.mean(0), resp - resp.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def to_weights(r, how):
    r = np.where(post, r, 0.0)  # post-onset only, as the identification itself
    if how == "weight":  # soft: each feature counts by its positive prediction accuracy
        return np.clip(r, 0, None)
    if how == "select":  # Kay et al. 2008: keep the best-predicted quarter
        cut = np.quantile(r[post], 1 - FIXED["select_share"])
        return (r >= cut).astype(np.float64)
    raise ValueError(how)

def similarity_w(pred, meas, w):
    # centred identification with features scaled by sqrt(w): the Pearson correlation of the scaled features (not a weighted Pearson correlation)
    cols = w > 0
    sw = np.sqrt(w[cols])
    p, m = pred[:, cols] - pred[:, cols].mean(0), meas[:, cols] - meas[:, cols].mean(0)
    p, m = p * sw, m * sw
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]

FEAT = {}
for feat in FIXED["features"]:
    with np.load(C.find_artifact("paper_alexnet_kpca1000.npz" if feat == "alexnet" else f"09m_{feat}_kpca1000.npz")) as f:
        ztr, zte = f["train"].astype(np.float64), f["test"].astype(np.float64)
    FEAT[feat] = (ztr, zte) + sources(ztr, zte)
print("sources fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")
rows = []
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    meas = bio_test[person].mean(1).reshape(N_TEST, -1).astype(np.float64)
    avgs = {feat: (s_tr[others].mean(0), s_te[others].mean(0).astype(np.float64)) for feat, (_, _, s_tr, s_te) in FEAT.items()}
    preds = {}
    for feat, (_, _, _, s_te) in FEAT.items():
        preds[("avg", 0, "-", feat)] = avgs[feat][1]
        preds[("own_full", "all", "-", feat)] = s_te[i].astype(np.float64)
    for d in FIXED["draws"]:
        for k in FIXED["ks"]:
            idx = order(person, d, k)
            parts = {}
            for feat, (ztr, zte, _, _) in FEAT.items():
                target, p = parts_for(person, idx, d, ztr, zte, *avgs[feat])
                parts.update({(name, feat): v for name, v in p.items()})
                preds[("h4b", k, d, feat)] = p["map"][1].reshape(N_TEST, -1)
                preds[("own_ridge", k, d, feat)] = p["own"][1].reshape(N_TEST, -1)
            for name, chosen in FIXED["stacks"].items():
                preds[("stack", k, d, name)] = stack(target, [parts[tuple(c)] for c in chosen])
    for (method, k, d, variant), pred in preds.items():
        t1, t5 = accuracy(similarity(pred, meas))
        rows.append({"person": person, "method": method, "k": k, "draw": d, "variant": variant, "top1": t1, "top5": t5,
                     "enc_r": encoding_r(pred, meas)})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(method, k, variant, metric="top1"):
    f = frame[(frame.method == method) & (frame.k.astype(str) == str(k)) & (frame.variant == variant)]
    return f.groupby("person")[metric].mean().reindex(people).to_numpy()
summary = {"post_hoc": "development; secondary by construction: combines earlier choices after they were seen; "
                       "THINGS-EEG2, held-out person, centred; chance 0.5% top-1", "results": {}, "contrasts": {}}
for f in FIXED["features"]:
    summary["results"][f"avg_{f}"] = {"top1": ci(per_person("avg", 0, f)), "enc_r": float(per_person("avg", 0, f, "enc_r").mean())}
    summary["results"][f"own_full_{f}"] = {"top1": ci(per_person("own_full", "all", f)), "enc_r": float(per_person("own_full", "all", f, "enc_r").mean())}
for k in FIXED["ks"]:
    block = {}
    for f in FIXED["features"]:
        block[f"map_{f}"] = ci(per_person("h4b", k, f))
        block[f"own_ridge_{f}"] = ci(per_person("own_ridge", k, f))
    for name in FIXED["stacks"]:
        block[f"stack_{name}"] = {"top1": ci(per_person("stack", k, name)), "top5": ci(per_person("stack", k, name, "top5")),
                                  "enc_r": ci(per_person("stack", k, name, "enc_r"))}
    summary["results"][f"k{k}"] = block
    summary["contrasts"][f"k{k}"] = {f"{name}_minus_alexnet": {"top1": ci(per_person("stack", k, name) - per_person("stack", k, "alexnet")),
                                                               "enc_r": ci(per_person("stack", k, name, "enc_r") - per_person("stack", k, "alexnet", "enc_r"))}
                                     for name in FIXED["stacks"] if name != "alexnet"}
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(figsize=(6.3, 3.8))
ks = FIXED["ks"]
for name, colour in zip(FIXED["stacks"], ("#2E5EAA", "#9DB4D9", "#C8102E")):
    ax.plot(ks, [summary["results"][f"k{k}"][f"stack_{name}"]["top1"]["mean"] for k in ks], "o-", color=colour, label=f"stack: {name}")
for f, style in zip(FIXED["features"], ("--", ":")):
    ax.plot(ks, [summary["results"][f"k{k}"][f"map_{f}"]["mean"] for k in ks], style, color="#7A7A7A", label=f"map only: {f}")
    ax.axhline(summary["results"][f"own_full_{f}"]["top1"]["mean"], ls=style, color="#E58A99", lw=1, label=f"own model, all images: {f}")
ax.set(xscale="log", xticks=ks, xticklabels=ks, xlabel="labelled images of the new person", ylabel="top-1 of 200 (%)",
       title="18j: THINGS-EEG2, held-out person (centred)")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({key: (val if key.startswith(("avg", "own")) else {n: round(v["top1"]["mean"] if "top1" in v else v["mean"], 1) for n, v in val.items()})
                  for key, val in summary["results"].items()}, indent=1, default=str)[:3000])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-18j-outputs", title="XSUB 18j decoding outputs",
                  files=[C.METRICS_DIR / f"18j_{x}" for x in ('summary.json', 'scores.csv', 'figure.png')],
                  include_dirs=[], message="development run")